In [4]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

spark = DatabricksSession.builder.getOrCreate()

df = spark.read.table("samples.nyctaxi.trips")

# Add average fare per mile column
# Handle division by zero by using when() condition
df = df.withColumn(
    "fare_per_mile", 
    F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance"))
    .otherwise(None)
)

df.show(5)

# Write the data to the specified table
df.write \
  .mode("overwrite") \
  .saveAsTable("main.dustinvannoy_dev.trips_with_fare_per_mile")

print("Data successfully written to main.dustinvannoy_dev.trips_with_fare_per_mile")

+--------------------+---------------------+-------------+-----------+----------+-----------+------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|     fare_per_mile|
+--------------------+---------------------+-------------+-----------+----------+-----------+------------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110| 5.714285714285714|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|5.7251908396946565|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018| 5.277777777777778|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|               5.0|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022| 7.115384615384615|
+--------------------+---------------------+-------------+-----------+----------+-----------+---

In [6]:
# Quick record count check
record_count = spark.sql("SELECT COUNT(*) as total_records FROM main.dustinvannoy_dev.trips_with_fare_per_mile").collect()[0]['total_records']
print(f"Total records in main.dustinvannoy_dev.trips_with_fare_per_mile: {record_count:,}")


Total records in main.dustinvannoy_dev.trips_with_fare_per_mile: 21,932


In [ ]:
# Demonstrate the new zone-based average fare per mile function
from ai_coding_tools.fare_calculator import add_avg_fare_per_mile_by_zones

# Load a sample of data for demonstration  
zone_demo_df = spark.read.table("samples.nyctaxi.trips").limit(500)

print("Sample data before adding zone-based averages:")
zone_demo_df.select("pickup_zip", "dropoff_zip", "fare_amount", "trip_distance").show(5)

# Add zone-based average fare per mile
df_with_zone_avg = add_avg_fare_per_mile_by_zones(zone_demo_df)

print("\nData with zone-based average fare per mile:")
df_with_zone_avg.select("pickup_zip", "dropoff_zip", "fare_amount", "trip_distance", "avg_fare_per_mile_by_zones").show(10)

# Show some route-specific statistics
print("\nTop 10 pickup/dropoff zone combinations by average fare per mile:")
route_stats = df_with_zone_avg.groupBy("pickup_zip", "dropoff_zip", "avg_fare_per_mile_by_zones") \
    .count() \
    .filter(F.col("avg_fare_per_mile_by_zones").isNotNull()) \
    .orderBy(F.desc("avg_fare_per_mile_by_zones")) \
    .limit(10)

route_stats.show()

# Compare individual vs zone-based averages for a specific route
print("\nComparison for routes from 10023:")
comparison_df = add_fare_per_mile_column(df_with_zone_avg)
comparison_df.filter(F.col("pickup_zip") == "10023") \
    .select("pickup_zip", "dropoff_zip", "fare_per_mile", "avg_fare_per_mile_by_zones") \
    .show(10)
